# 👟 E-Ticaret Satış Danışmanı Chatbotu - Query Intent Routing
Gemini (`google-genai`) + ChromaDB. Router her sorguyu 3 yoldan birine yönlendirir:
**A** Metadata filtre (vektörsüz) · **B** Semantik arama · **C** LLM karşılaştırma.

**Ön hazırlık**
1. Bu klasördeki `.py` dosyalarını Drive'a `MyDrive/shoe-advisor/` içine yükle.
2. Colab sol menü 🔑 **Secrets** → `GEMINI_API_KEY` ekle ve **Notebook access** anahtarını aç.

In [ ]:
!pip install -q google-genai chromadb pydantic datasets

In [ ]:
# Drive bağlantısı ve proje yolu
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/shoe-advisor'
DATA_PATH   = f'{PROJECT_DIR}/data/shoes.json'
CHROMA_DIR  = f'{PROJECT_DIR}/data/chroma'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
print(sorted(f for f in os.listdir(PROJECT_DIR) if f.endswith('.py')))

In [ ]:
# Gemini istemcisi (anahtar Colab Secrets'tan okunur) + hızlı sağlık kontrolü
import logging, importlib
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(message)s')
import gemini_client, vector_store, router, advisor, data_loader
for m in (gemini_client, vector_store, router, advisor, data_loader): importlib.reload(m)

from gemini_client import GeminiClient
llm = GeminiClient()
print('Embedding boyutu:', len(llm.embed_query('test')))
print('LLM yanıtı     :', llm.generate_text('Tek kelime ile selam ver.', temperature=0))
print('Aktif LLM      :', llm._active_llm)

In [ ]:
# 1) VERİ: Hugging Face'ten streaming ile ilk 300 ayakkabı (varsa Drive'daki dosya kullanılır)
import json
from pathlib import Path
import data_loader

if Path(DATA_PATH).exists():
    products = json.load(open(DATA_PATH, encoding='utf-8'))
    print('Drive\'dan yüklendi:', len(products))
else:
    # USD/TRY kurunu güncel değerle değiştir
    products = data_loader.collect_shoes(limit=300, max_scan=500_000, usd_try=40.0, seed=42)
    data_loader.save_json(products, Path(DATA_PATH))
data_loader.print_summary(products)

In [ ]:
# 2) İNDEKS: Gemini embedding (25'lik batch) -> ChromaDB (Drive'da kalıcı, kaldığı yerden devam eder)
from vector_store import ShoeVectorStore
store = ShoeVectorStore(llm, persist_dir=CHROMA_DIR)
print('Koleksiyondaki ürün:', store.index_products(products, batch_size=25, pause=1.5))
print('Markalar (ilk 15):', store.list_brands()[:15])

In [ ]:
# 3) PIPELINE + güzel çıktı yardımcısı
from advisor import ShoeAdvisorPipeline, format_price
from IPython.display import Markdown, display
pipe = ShoeAdvisorPipeline(llm, store, top_k=5)

def show(q):
    out = pipe.ask(q)
    print('='*90); print('SORU   :', q)
    print('YOL    :', out['intent'], f"({out['router_source']})"); print('GEREKÇE:', out['reasoning'])
    print('SLOTLAR:', out['slots'])
    if out.get('semantic_query'): print('SEM.SORGU:', out['semantic_query'])
    if out.get('applied_filters'): print('FİLTRELER:', out['applied_filters'])
    if out['notes']: print('NOTLAR :', out['notes'])
    print('-'*90)
    for i, p in enumerate(out['products'], 1):
        sim = f" | benzerlik {p['similarity']}" if 'similarity' in p else ''
        print(f"{i}. {p['title'][:70]} | {format_price(p['price'])} | {p['rating']:.1f}★ | beden {p['sizes']}{sim}")
    display(Markdown(out['answer']))
    return out

### Senaryo 1 - Filtre (PATH A: vektör araması yok)

In [ ]:
_ = show('1500 TL altı 42 numara spor ayakkabı')

### Senaryo 2 - Semantik (PATH B: kosinüs benzerliği)

In [ ]:
_ = show('Bütün gün ayakta çalışıyorum, tabanı yumuşak ve ayağı terletmeyen bir model')

### Senaryo 3 - Karşılaştırma (PATH C): Senaryo 2'deki ilk iki ürün üzerinden

In [ ]:
_ = show('Listedeki ilk iki ürünü karşılaştır, uzun süre ayakta çalışmak için hangisi daha iyi?')

### (Opsiyonel) Serbest sohbet

In [ ]:
while True:
    q = input('Sen (çıkmak için boş bırak): ').strip()
    if not q: break
    show(q)